# Deterministic Evaluation for AI Agents - Evaluate Agent Tool Execution Outcome
Using AgentCore Custom Code Evaluator to evaluate agent tool execution outcome. Basically check if tool updated the backend system or not.

In this **minimal, end-to-end** sample:
1. **Amazon DynamoDB** table works as backend for agent tool.
2. A **Strands agent** with one tool `save_record` that writes a record to Amazon DynamoDB table.
3. An **AgentCore custom code-based evaluator** that reads the agent's log with session and span details and **checks the record exists in Amazon DynamoDB Table** → `PASS` / `FAIL`.

## ⚠️ Before you run — account prerequisites

The notebook creates and tears down **all** its own resources (DynamoDB table, IAM roles, Lambda, AgentCore Runtime, evaluator). Few prerequisite to run notebook successfully:

1. **AWS credentials / IAM Role configured** with permission to use DynamoDB, IAM, Lambda, and Bedrock AgentCore.
2. **Bedrock model access** — enable the agent's model in the Bedrock console for your `REGION`.
3. **CloudWatch Transaction Search + Runtime Observability** — required evaluation.

**File layout:** keep this notebook in the same folder as `agent_runtime.py`, `evaluator_lambda.py`, and `requirements.txt` (they ship alongside it). The deploy and Lambda-packaging cells reference these files by relative name, so run the notebook from that folder.

**Placeholders to set:** `REGION` (Step 1). Everything else — ARNs, role names, session IDs — is captured automatically as the notebook runs.

In [ ]:
REGION = "us-east-1"        # <-- your region

## Step 0 — Install dependencies

Run this once. `strands-agents` = the agent framework; `bedrock-agentcore` = the Runtime SDK (`BedrockAgentCoreApp`); `bedrock-agentcore-starter-toolkit` = the `agentcore` CLI used to deploy; `boto3` = AWS SDK for DynamoDB / Lambda / evaluator calls.

In [ ]:
%pip install -q strands-agents bedrock-agentcore bedrock-agentcore-starter-toolkit boto3 aws-opentelemetry-distro
print("Dependencies installed. Restart the kernel before continuing.")

## Step 1 — Create the DynamoDB table
Create a DynamoDB which works like backend for the Agent Tool. When Agent Tool executes, it write a record into this table.

In [ ]:
import boto3, time

TABLE  = "agent_records"

ddb = boto3.resource("dynamodb", region_name=REGION)

def create_table():
    existing = [t.name for t in ddb.tables.all()]
    if TABLE in existing:
        print("Table already exists:", TABLE); return
    ddb.create_table(
        TableName=TABLE,
        KeySchema=[{"AttributeName": "id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "id", "AttributeType": "S"}],
        BillingMode="PAY_PER_REQUEST",
    ).wait_until_exists()
    print("Created table:", TABLE)

create_table()

## Step 2 — Define the tool and the Strands agent

Creating an agent with a tool. When agent tool executes, it writes a record into the DynamoDB table. Also running the agent to test if tool works.

In [ ]:
import uuid, datetime
from strands import Agent, tool
from strands.models import BedrockModel

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)

table = ddb.Table(TABLE)

@tool
def save_record(content: str) -> str:
    "Save a text record to the database. Returns the new record_id."
    record_id = str(uuid.uuid4())
    table.put_item(Item={
        "id": record_id,
        "content": content,
        "created_at": datetime.datetime.utcnow().isoformat(),
    })
    return record_id

# Strands agent with the tool. Model defaults to Bedrock Claude Sonnet.
agent = Agent(
    model=model,
    tools=[save_record],
    system_prompt=("You save records for the user. When asked to remember/log something, "
                   "call save_record and then tell the user the record_id."),
)

# Local smoke test (needs AWS creds + Bedrock model access)
result = agent("Please remember: quarterly review is on Friday.")
print(result.message)

## Step 3 — Deploy to AgentCore Runtime

Copied the agent code above to  `agent_runtime.py` and now deploying the agent to AgentCore runtime using agentcore starter toolkit.

#### 3a. Deploy Agent
Deploy agent keeping most of the configurations such auto create role and auto create ECR to the default.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import os, glob

# Guard: if a previous runtime was deleted, its stale local config makes the toolkit try to
# UPDATE a non-existent runtime (ResourceNotFoundException). Remove stale config for a clean create.
for _f in glob.glob(".bedrock_agentcore*"):
    try: os.remove(_f)
    except OSError: pass

agent_name = "recordagent"
agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="agent_runtime.py",
    auto_create_ecr=True,
    auto_create_execution_role=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
)

launch_result = agentcore_runtime.launch()   # CodeBuild by default, no Docker
print("Launch result:", launch_result)

# Agent Runtime ARN (attribute name varies by toolkit version)
AGENT_ARN = (
    getattr(launch_result, "agent_arn", None)
    or getattr(launch_result, "agent_runtime_arn", None)
    or (launch_result.get("agent_arn") if isinstance(launch_result, dict) else None)
)
# agent_id is the last ARN segment (or read from launch_result); needed for on-demand eval in Step 5.
agent_id = (
    getattr(launch_result, "agent_id", None)
    or (AGENT_ARN.split("/")[-1] if AGENT_ARN else None)
)
print("AGENT_ARN =", AGENT_ARN or "<see launch_result above>")
print("agent_id  =", agent_id)


#### 3b. DynamoDB table write access for agent execution role

The toolkit auto-created agent role and the role by default can't write to the DynamoDB table. Add permission to the role to be able to write to DynamoDB table.

In [ ]:
# 3c. Attach dynamodb:PutItem to the auto-created Runtime execution role
import boto3, json, os, time

ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
iam = boto3.client("iam")
table_arn = f"arn:aws:dynamodb:{REGION}:{ACCOUNT}:table/{TABLE}"

# The toolkit writes the execution role ARN into .bedrock_agentcore.yaml
RUNTIME_ROLE_ARN = None
try:
    import yaml
    cfg = yaml.safe_load(open(os.path.join(os.getcwd(), ".bedrock_agentcore.yaml")))
    def _find_role(o):
        if isinstance(o, dict):
            for v in o.values():
                if isinstance(v, str) and v.startswith("arn:aws:iam::") and ":role/" in v and "Runtime" in v:
                    return v
                r = _find_role(v)
                if r: return r
        elif isinstance(o, list):
            for v in o:
                r = _find_role(v)
                if r: return r
        return None
    RUNTIME_ROLE_ARN = _find_role(cfg)
except Exception as e:
    print("Could not auto-read role ARN:", e)

RUNTIME_ROLE_NAME = RUNTIME_ROLE_ARN.split(":role/")[-1]
print("Runtime execution role:", RUNTIME_ROLE_NAME)

iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="ddb-write-records",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{"Effect": "Allow", "Action": ["dynamodb:PutItem"], "Resource": table_arn}],
    }),
)
print("Attached dynamodb:PutItem on", table_arn)
time.sleep(10)  # IAM propagation before first invoke


#### 3c. Invoke the deployed agent
Check if deployed agent working and the tool is able to insert record into DynamoDB table.
We also keep session id of the execution in **SESSION_ID** variable because later we use it to run custom evaluator for agent tool evaluation

In [ ]:
# 3d. Invoke the deployed agent
import boto3, json, uuid

rt = boto3.client("bedrock-agentcore", region_name=REGION)

SESSION_ID = "session-" + uuid.uuid4().hex   
resp = rt.invoke_agent_runtime(
    agentRuntimeArn=AGENT_ARN,
    runtimeSessionId=SESSION_ID,
    payload=json.dumps({"prompt": "Log: renewed the support contract"}).encode(),
)
print("SESSION_ID =", SESSION_ID)
print(json.loads(resp["response"].read()))


## Step 4 — Create AgentCore Custom Evaluator (code-based based)

We create Amazon Bedrock AgentCore custom evaluator which will evaluate the agent to check if the agent tool has created record in the DynamoDB table.
There are three steps:
4a) Create IAM role for Lambda function so that Lambda function can read CloudWatch Log for Agent and also read DynamoDB table.
4b) Create Lambda function with IAM Role. Lambda function read tool execution details from CloudWatch log and checks existance of record in the DynamoDB table.
4c) Create Customer Evaluator using Lambda Function.

#### 4a. Create IAM role for Lambda function with required permissions

In [ ]:
import boto3, json, time

iam = boto3.client("iam")
ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
ROLE_NAME = "lambda-eval-role"
table_arn = f"arn:aws:dynamodb:{REGION}:{ACCOUNT}:table/{TABLE}"

trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}

try:
    iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust),
        Description="Execution role for the AgentCore record-verification evaluator Lambda",
    )
    print("Created role:", ROLE_NAME)
    created = True
except iam.exceptions.EntityAlreadyExistsException:
    print("Role already exists:", ROLE_NAME)
    created = False

# Basic execution (CloudWatch Logs)
iam.attach_role_policy(
    RoleName=ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)
# Inline: read the records table
iam.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="ddb-read-records",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["dynamodb:GetItem"],
            "Resource": table_arn,
        }],
    }),
)

LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT}:role/{ROLE_NAME}"
print("LAMBDA_ROLE_ARN =", LAMBDA_ROLE_ARN)

if created:
    print("Waiting ~10s for IAM role propagation before creating the Lambda...")
    time.sleep(10)


#### 4b. Create the Lambda function
Lambda function code is in the file `evaluator_lambda.py`

In [ ]:
import boto3, io, zipfile

# Build the zip in memory from the lambda function code file
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.write("evaluator_lambda.py")
buf.seek(0)

lam = boto3.client("lambda", region_name=REGION)
try:
    r = lam.create_function(
        FunctionName="verify-record-saved",
        Runtime="python3.12",
        Handler="evaluator_lambda.lambda_handler",
        Role=LAMBDA_ROLE_ARN,
        Code={"ZipFile": buf.read()},
        Timeout=60,
    )
    LAMBDA_ARN = r["FunctionArn"]
except lam.exceptions.ResourceConflictException:
    buf.seek(0)
    lam.update_function_code(FunctionName="verify-record-saved", ZipFile=buf.read())
    LAMBDA_ARN = lam.get_function(FunctionName="verify-record-saved")["Configuration"]["FunctionArn"]

print("LAMBDA_ARN =", LAMBDA_ARN)


#### 4C. Create Customer Evaluator using the Lambda function 

In [ ]:
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)

resp = ctrl.create_evaluator(
    evaluatorName="VerifyRecordSaved",
    level="TRACE",                       # TRACE | TOOL_CALL | SESSION
    evaluatorConfig={
        "codeBased": {
            "lambdaConfig": {
                "lambdaArn": LAMBDA_ARN,
                "lambdaTimeoutInSeconds": 60,
            }
        }
    },
)
evaluator_id = resp["evaluatorId"]
print("Evaluator:", evaluator_id)


## Step 5 — Run on-demand evaluation

We already run the agent earlier with session id stored in **SESSION_ID**. We now run evaluation on-demand using custom evaluator for the session run.
You will also see outcome of the evaluation.

> **Prereq:** `Evaluation().run()` reads spans from CloudWatch, so **Transaction Search** must be enabled and `aws-opentelemetry-distro` must be in `requirements.txt`. Spans take 1–2 min to index after the run — if results come back empty, wait and re-run this cell.

In [ ]:
### CHECK CLOUDWATCH LOG TO SEE IF THIS SESSION ID IS PUBLISHED THERE ####
print(SESSION_ID)

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation

eval_client = Evaluation()
custom_results = eval_client.run(
    agent_id=agent_id,          
    session_id=SESSION_ID,      
    evaluators=[evaluator_id],  
)

successful = custom_results.get_successful_results()
failed = custom_results.get_failed_results()
print(f"Successful: {len(successful)} | Failed: {len(failed)}")

for r in successful:
    print(f"\n{r.evaluator_name}: {r.label} (value={r.value})")
    if r.explanation:
        print("  ", r.explanation)


## Step 6 — Clean up all resources

Removes everything this notebook created so you don't leave billable resources behind:
the evaluator, the Lambda, the AgentCore Runtime, and the DynamoDB table.

In [ ]:
import boto3

REGION = REGION if "REGION" in dir() else "us-east-1"

def _try(desc, fn):
    try:
        fn()
        print("Deleted:", desc)
    except Exception as e:
        print(f"Skip {desc}: {e.__class__.__name__} - {e}")

# 1. Delete the AgentCore evaluator
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)
if "evaluator_id" in dir():
    _try(f"evaluator {evaluator_id}", lambda: ctrl.delete_evaluator(evaluatorId=evaluator_id))

# 2. Delete the AgentCore Runtime (list, match by name, delete)
def _delete_runtime():
    for rt in ctrl.list_agent_runtimes().get("agentRuntimes", []):
        if rt.get("agentRuntimeName", "").startswith("recordagent"):
            ctrl.delete_agent_runtime(agentRuntimeId=rt["agentRuntimeId"])
            print("  removed runtime:", rt["agentRuntimeName"])
_try("AgentCore runtime(s) 'recordagent*'", _delete_runtime)

# 3. Delete the evaluator Lambda
lam = boto3.client("lambda", region_name=REGION)
_try("Lambda verify-record-saved",
     lambda: lam.delete_function(FunctionName="verify-record-saved"))

# 4. Delete the DynamoDB table
ddb_client = boto3.client("dynamodb", region_name=REGION)
_try(f"DynamoDB table {TABLE}",
     lambda: ddb_client.delete_table(TableName=TABLE))

# 5. Delete the IAM roles we created (detach inline + managed policies first)
iam = boto3.client("iam")
def _delete_role(role_name):
    def _fn():
        for pol in iam.list_role_policies(RoleName=role_name).get("PolicyNames", []):
            iam.delete_role_policy(RoleName=role_name, PolicyName=pol)
        for ap in iam.list_attached_role_policies(RoleName=role_name).get("AttachedPolicies", []):
            iam.detach_role_policy(RoleName=role_name, PolicyArn=ap["PolicyArn"])
        iam.delete_role(RoleName=role_name)
    return _fn

# We own lambda-eval-role -> delete it fully.
_try("IAM role lambda-eval-role", _delete_role("lambda-eval-role"))

# The Runtime execution role was auto-created by the toolkit (AmazonBedrockAgentCoreSDKRuntime-*).
# The toolkit manages its lifecycle; if you deployed via the toolkit, use `agentcore destroy`
# (or delete the Runtime + role in the console). We only remove the inline PutItem policy we added.
if "RUNTIME_ROLE_NAME" in dir():
    _try(f"inline ddb-write policy on {RUNTIME_ROLE_NAME}",
         lambda: iam.delete_role_policy(RoleName=RUNTIME_ROLE_NAME, PolicyName="ddb-write-records"))

# 6. Remove the toolkit's local config so a fresh run creates a NEW runtime
#    (otherwise configure()/launch() tries to UPDATE the deleted runtime -> ResourceNotFoundException).
import os, glob
def _clear_toolkit_state():
    removed = []
    for f in [".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"] + glob.glob(".bedrock_agentcore*"):
        if os.path.exists(f):
            os.remove(f); removed.append(f)
    print("  removed local files:", removed or "none")
_try("toolkit local config (.bedrock_agentcore.yaml, etc.)", _clear_toolkit_state)

print("\nCleanup complete.")
